In [ ]:
import cv2
import numpy as np
import glob
import os
from matplotlib import pyplot as plt
from IPython.display import clear_output

In [ ]:
# Parámetros globales
IMAGE_FOLDER = "M-30-HD/" # input
ROI_FILE = "roi.npy"      # roi

OUTPUT_MOG2 = "mog2/"     # ruta para guardar los fotogramas con sustracción del fondo
OUTPUT_VIS = "tracking/"  # ruta para guardar los fotogramas con los vehículos trackeados

MIN_AREA = 500            # área (píxeles al cuadrado) mínima para aceptar un contorno como vehículo
MAX_DISTANCE = 50         # distancia (píxeles) máxima para asociar detecciones al mismo objeto
MAX_LOST = 2              # máx. fotogramas seguidos sin ver un objeto antes de eliminarlo
WARMUP_FRAMES = 120       # fotogramas iniciales para estabilizar MOG2 (warm-up)

next_object_id = 1        # inicialización ID incremental para nuevos objetos detectados

# Crear las rutas de salida si no existen
os.makedirs(OUTPUT_MOG2, exist_ok=True)
os.makedirs(OUTPUT_VIS, exist_ok=True)

In [ ]:
# Clase para almacenar la información de cada objeto trackeado
class TrackedObject:
    def __init__(self, object_id, centroid, bbox):
        self.id = object_id                     # identificador único
        self.centroid = centroid                # centroide actual del objeto
        self.bbox = bbox                        # bounding box (x, y, w, h)
        self.lost = 0                           # fotogramas consecutivos sin detectar
        self.color = tuple(np.random.randint(0, 255, size=3).tolist())  # color aleatorio para dibujar
        self.flow_vector = (0, 0)               # vector medio de flujo óptico en su región

# Calcula el centroide a partir de una bounding box
def centroid_from_bbox(x, y, w, h):
    return (int(x + w/2), int(y + h/2))

# Distancia euclídea entre dos puntos (píxeles)
def euclidean_distance(p1, p2):
    return np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

# Fusiona cajas solapadas en una sola (para evitar múltiples detecciones del mismo objeto)
def merge_overlapping_boxes(boxes):
    merged = []
    for box in boxes:
        x, y, w, h = box
        added = False
        for i, (mx, my, mw, mh) in enumerate(merged):
            # Comprobación de solapamiento entre cajas
            if not (x + w < mx or mx + mw < x or y + h < my or my + mh < y):
                # Fusionar ambas cajas en una más grande
                nx = min(x, mx)
                ny = min(y, my)
                nw = max(x + w, mx + mw) - nx
                nh = max(y + h, my + mh) - ny
                merged[i] = (nx, ny, nw, nh)
                added = True
                break
        if not added:
            merged.append(box)
    return merged


In [ ]:
# Carga y ordena todos los fotogramas .jpg de la carpeta
def load_images(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    if not files:
        raise RuntimeError("No se encontraron imágenes en la carpeta.")
    return files

# Carga la máscara de la región de interés (ROI) desde roi.npy
def load_roi(path):
    if not os.path.exists(path):
        raise RuntimeError("No se encontró roi.npy")
    return np.load(path)

In [ ]:
# Crea el sustractor de fondo MOG2 con parámetros ajustados
def create_mog2():
    return cv2.createBackgroundSubtractorMOG2(
        history=200,        # nº de fotogramas usados para el modelo de fondo (α = 1/200)
        varThreshold=25,    # umbral de coincidencia (k=5 desviaciones estándar)
        detectShadows=False # desactiva detección de sombras (evita falsos positivos)
    )

In [ ]:
# Aplica MOG2 al fotograma y guarda la máscara generada
def process_mog2(frame, fgbg, idx):
    fgmask = fgbg.apply(frame)  # máscara de primer plano (MOG2)

    # Guardar la máscara original generada por MOG2
    out_path = os.path.join(OUTPUT_MOG2, f"mog2_{idx:04d}.png")
    cv2.imwrite(out_path, fgmask)

    return fgmask  # devolver máscara para el postprocesado

In [ ]:
# Postprocesado de la máscara MOG2 (limpieza + ROI + morfología)
def postprocess_mask(fgmask, roi_mask):
    # Binarizar la máscara (descartar valores bajos de MOG2)
    _, fgmask_bin = cv2.threshold(fgmask, 200, 255, cv2.THRESH_BINARY)

    # Reducir ruido impulsivo
    fgmask_bin = cv2.medianBlur(fgmask_bin, 5)

    # Aplicar la región de interés (ROI)
    fgmask_bin = cv2.bitwise_and(fgmask_bin, roi_mask)

    # Eliminar ruido pequeño (apertura morfológica)
    kernel_noise = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)) 
    fgmask_bin = cv2.morphologyEx(fgmask_bin, cv2.MORPH_OPEN, kernel_noise)

    # Expandir ligeramente las regiones detectadas
    kernel_small = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    fgmask_bin = cv2.dilate(fgmask_bin, kernel_small, iterations=2)

    # Cerrar huecos internos en los vehículos (cierre morfológico)
    kernel_big = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    fgmask_bin = cv2.morphologyEx(fgmask_bin, cv2.MORPH_CLOSE, kernel_big)

    return fgmask_bin

In [ ]:
# Detecta las cajas delimitadoras (bounding boxes) a partir de la máscara binaria
def detect_boxes(fgmask_bin):
    # Encontrar contornos externos en la máscara
    contours, _ = cv2.findContours(fgmask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    raw_boxes = []

    for cnt in contours:
        # Ignorar contornos demasiado pequeños (ruido)
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        # Obtener bounding box del contorno
        raw_boxes.append(cv2.boundingRect(cnt))

    # Fusionar cajas solapadas (según el criterio definido)
    return merge_overlapping_boxes(raw_boxes)

In [ ]:
# Actualiza el estado de los objetos trackeados y asocia nuevas detecciones
def update_tracks(detections, tracked_objects, flow, width, height):
    global next_object_id

    # Marcar todos los objetos como "no vistos" en este frame
    for obj in tracked_objects:
        obj.lost += 1

    # Intentar asociar cada detección con un objeto existente
    for det_centroid, det_bbox in detections:
        best_obj = None
        best_dist = float("inf")

        # Buscar el objeto más cercano dentro del umbral permitido
        for obj in tracked_objects:
            dist = euclidean_distance(det_centroid, obj.centroid)
            if dist < best_dist and dist < MAX_DISTANCE:
                best_dist = dist
                best_obj = obj

        if best_obj:
            # Actualizar objeto existente
            best_obj.centroid = det_centroid
            best_obj.bbox = det_bbox
            best_obj.lost = 0  # resetea contador de "no visto"

            # Calcular flujo óptico medio dentro de la bounding box
            x, y, w, h = det_bbox
            roi_flow = flow[y:y+h, x:x+w]
            if roi_flow.size > 0:
                best_obj.flow_vector = (
                    np.mean(roi_flow[..., 0]),
                    np.mean(roi_flow[..., 1])
                )
        else:
            # Crear un nuevo objeto si no se ha encontrado correspondencia
            tracked_objects.append(TrackedObject(next_object_id, det_centroid, det_bbox))
            next_object_id += 1

    # Eliminar objetos que llevan demasiado tiempo sin ser detectados
    return [obj for obj in tracked_objects if obj.lost <= MAX_LOST]

In [ ]:
# Dibuja las cajas, IDs y vectores de movimiento, y guarda el fotograma resultante
def draw_and_save(frame, tracked_objects, roi_mask, idx):
    vis = frame.copy()

    # Atenuar zonas fuera de la ROI para destacar la región útil
    vis[roi_mask == 0] = (vis[roi_mask == 0] * 0.3).astype(np.uint8)

    for obj in tracked_objects:
        x, y, w, h = obj.bbox
        cx, cy = obj.centroid

        # Dibujar bounding box e ID del objeto
        cv2.rectangle(vis, (x, y), (x + w, y + h), obj.color, 2)
        cv2.putText(vis, f"ID {obj.id}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, obj.color, 2)

        # Dibujar velocidad estimada mediante flujo óptico
        vx, vy = obj.flow_vector
        speed = abs(vx) + abs(vy)
        cv2.putText(vis, f"v={speed:.2f}", (x, y + h + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, obj.color, 2)

        # Dibujar flecha indicando dirección del movimiento
        end_point = (int(cx + vx * 5), int(cy + vy * 5))
        cv2.arrowedLine(vis, (cx, cy), end_point, (0, 255, 0), 2)

    # Guardar fotograma visualizado
    out_path = os.path.join(OUTPUT_VIS, f"vis_{idx:04d}.png")
    cv2.imwrite(out_path, vis)

    return vis

In [ ]:
# Bucle principal del sistema: carga datos, aplica MOG2, detecta, trackea y visualiza
def main(use_imshow=True):
    image_files = load_images(IMAGE_FOLDER)   # lista de fotogramas
    roi_mask = load_roi(ROI_FILE)             # máscara de región de interés
    fgbg = create_mog2()                      # inicializar MOG2

    tracked_objects = []                      # lista de objetos en seguimiento
    prev_gray = cv2.cvtColor(cv2.imread(image_files[0]), cv2.COLOR_BGR2GRAY)

    frame_count = 0

    # Preparar ventana si se usa visualización en tiempo real
    if use_imshow:
        cv2.namedWindow("Tracking de coches", cv2.WINDOW_NORMAL)

    # Recorrer todos los fotogramas (excepto el primero)
    for idx, img_path in enumerate(image_files[1:], start=1):
        frame = cv2.imread(img_path)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_count += 1

        # Fase de calentamiento para estabilizar el modelo de fondo
        if frame_count < WARMUP_FRAMES:
            fgbg.apply(frame)
            prev_gray = gray.copy()
            continue

        # 1. Sustracción de fondo (MOG2)
        fgmask = process_mog2(frame, fgbg, idx)
        fgmask_bin = postprocess_mask(fgmask, roi_mask)

        # 2. Flujo óptico (antes de detectar cajas)
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # 3. Detección de vehículos
        boxes = detect_boxes(fgmask_bin)
        detections = [(centroid_from_bbox(*b), b) for b in boxes]

        # 4. Actualizar tracking con detecciones + flujo óptico
        tracked_objects = update_tracks(
            detections, tracked_objects, flow,
            frame.shape[1], frame.shape[0]
        )

        # 5. Dibujar resultados y guardar fotograma
        vis = draw_and_save(frame, tracked_objects, roi_mask, idx)

        prev_gray = gray.copy()

        # ---------------------------
        # MODO 1: Visualización OpenCV
        # ---------------------------
        if use_imshow:
            cv2.imshow("Tracking de coches", vis)
            if cv2.waitKey(20) & 0xFF == 27:  # tecla ESC para salir
                break

        # ---------------------------
        # MODO 2: Visualización en notebook
        # ---------------------------
        else:
            clear_output(wait=True)
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.axis("off")
            plt.show()

    # Cerrar ventana si se usó imshow
    if use_imshow:
        cv2.destroyAllWindows()


In [ ]:
main(use_imshow=True)